In [1]:
# =========================
# 1: IMPORTS
# =========================
# Estas librerías permiten automatizar el navegador, esperar elementos,
# manejar pausas y organizar los datos capturados.

import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
# =========================
#2: CONFIGURACIÓN GENERAL
# =========================
# Aquí definimos metadatos del scraper y la URL objetivo.

NOMBRE_GRUPO = "YHadfeg_Amazon"
BUSQUEDA = "laptop"
URL_OBJETIVO = f"https://www.amazon.com/s?k={BUSQUEDA}"
LIMITE_PRODUCTOS = 3  # Puedes subirlo luego a 8 o 10

In [3]:
# =========================
# 3: NAVEGADOR VISIBLE
# =========================
# Esta función crea el navegador Chrome visible en el contenedor.
# Así se puede revisar en VNC si Amazon muestra captcha o bloqueo.

def iniciar_navegador_visible():
    options = Options()

    # Ruta del navegador dentro del contenedor Docker
    options.binary_location = "/usr/bin/google-chrome"

    # Opciones de estabilidad para Docker
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1366,768")

    # Simula un navegador real
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )

    driver = webdriver.Chrome(service=Service(), options=options)
    return driver

In [5]:
# =========================
# 4: FUNCIONES AUXILIARES
# =========================
# Estas funciones ayudan a extraer texto, limpiar el precio
# y detectar si la página fue bloqueada.

def extraer_texto_seguro(elemento, selectores_css):
    """
    Intenta obtener el texto de un subelemento probando varios selectores.
    Si no encuentra ninguno, devuelve None.
    """
    for selector in selectores_css:
        try:
            texto = elemento.find_element(By.CSS_SELECTOR, selector).text.strip()
            if texto:
                return texto
        except Exception:
            continue
    return None


def limpiar_precio(precio_texto):
    """
    Convierte un precio en texto a número flotante.
    """
    if not precio_texto:
        return None

    texto = precio_texto.replace("$", "").replace(",", "").strip()

    try:
        return float(texto)
    except Exception:
        return None


def detectar_bloqueo(driver):
    """
    Revisa señales de captcha o bloqueo.
    """
    titulo = driver.title.lower()
    html = driver.page_source.lower()

    if "robot" in titulo or "captcha" in titulo:
        return True

    if "enter the characters you see below" in html:
        return True

    if "sorry, we just need to make sure you're not a robot" in html:
        return True

    return False

In [6]:
# =========================
# 5: DETALLE DEL PRODUCTO
# =========================
# Estas funciones abren cada producto en una nueva pestaña,
# capturan datos más completos y vuelven al listado.

def abrir_producto_en_nueva_pestana(driver, url_producto):
    driver.execute_script("window.open(arguments[0], '_blank');", url_producto)
    driver.switch_to.window(driver.window_handles[-1])


def cerrar_pestana_actual(driver):
    driver.close()
    driver.switch_to.window(driver.window_handles[0])


def capturar_detalle_producto(driver):
    """
    Captura categoría, marca, disponibilidad y descripción.
    """
    detalle = {
        "categoria": None,
        "marca": None,
        "disponibilidad": None,
        "descripcion": None,
    }

    try:
        WebDriverWait(driver, 8).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(2)

        # -------- CATEGORÍA --------
        try:
            categorias = driver.find_elements(
                By.CSS_SELECTOR,
                "#wayfinding-breadcrumbs_feature_div ul li"
            )
            categorias_limpias = [c.text.strip() for c in categorias if c.text.strip()]
            if categorias_limpias:
                detalle["categoria"] = " > ".join(categorias_limpias)
        except Exception:
            pass

        # -------- MARCA --------
        posibles_selectores_marca = [
            "#bylineInfo",
            "tr.po-brand td.a-span9 span",
            "tr.po-brand td.a-span9",
            "tr.a-spacing-small td.a-span9 span",
        ]

        marca = None
        for selector in posibles_selectores_marca:
            try:
                marca = driver.find_element(By.CSS_SELECTOR, selector).text.strip()
                if marca:
                    break
            except Exception:
                continue

        if marca:
            detalle["marca"] = (
                marca.replace("Brand:", "")
                     .replace("Visit the", "")
                     .replace("Store", "")
                     .strip()
            )

        # -------- DISPONIBILIDAD --------
        posibles_selectores_stock = [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div",
            "#outOfStock span",
        ]

        for selector in posibles_selectores_stock:
            try:
                stock = driver.find_element(By.CSS_SELECTOR, selector).text.strip()
                if stock:
                    detalle["disponibilidad"] = stock
                    break
            except Exception:
                continue

        # -------- DESCRIPCIÓN --------
        posibles_selectores_descripcion = [
            "#feature-bullets ul li span",
            "#productDescription span",
            "#productFactsDesktopExpander p span",
        ]

        descripciones = []
        for selector in posibles_selectores_descripcion:
            try:
                elementos = driver.find_elements(By.CSS_SELECTOR, selector)
                textos = [e.text.strip() for e in elementos if e.text.strip()]
                if textos:
                    descripciones = textos[:3]
                    break
            except Exception:
                continue

        if descripciones:
            detalle["descripcion"] = " | ".join(descripciones)

    except Exception:
        pass

    return detalle

In [7]:
# =========================
# 6: FUNCIÓN PRINCIPAL
# =========================
# Esta función ejecuta el scraping completo y retorna
# una lista de diccionarios, para luego integrarlo con el main del grupo.

def ejecutar_extraccion():
    driver = None
    datos_finales = []

    try:
        # 1. Iniciar navegador
        driver = iniciar_navegador_visible()
        print("Abriendo Amazon...")
        driver.get(URL_OBJETIVO)

        # 2. Pausa para intervención humana
        print("\nACCIÓN REQUERIDA:")
        print("1. Ve a http://localhost:6080/vnc.html")
        print("2. Revisa si Amazon muestra captcha o validación.")
        print("3. Si aparece, resuélvelo manualmente.")
        print("4. Cuando veas el listado de productos, vuelve aquí.")
        input("\nPresiona ENTER para continuar con la extracción...")

        # 3. Verificar bloqueo
        if detectar_bloqueo(driver):
            print("Bloqueo detectado. No se continuará con la extracción.")
            return datos_finales

        print("Sin bloqueo evidente. Iniciando scraping...")

        # 4. Esperar listado
        WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div.s-result-item[data-component-type='s-search-result']")
            )
        )
        time.sleep(2)

        productos = driver.find_elements(
            By.CSS_SELECTOR,
            "div.s-result-item[data-component-type='s-search-result']"
        )

        print(f"Bloques encontrados en el listado: {len(productos)}")

        # 5. Recorrer productos
        for i, producto in enumerate(productos[:LIMITE_PRODUCTOS], start=1):
            try:
                # -------- NOMBRE --------
                nombre_producto = extraer_texto_seguro(producto, [
                    "h2 span",
                    "a h2 span",
                ])

                # -------- LINK --------
                url_producto = None
                try:
                    enlace = producto.find_element(By.CSS_SELECTOR, "h2 a")
                    url_producto = enlace.get_attribute("href")
                except Exception:
                    pass

                # -------- PRECIO --------
                precio_entero = extraer_texto_seguro(producto, [
                    "span.a-price-whole"
                ])
                precio_decimal = extraer_texto_seguro(producto, [
                    "span.a-price-fraction"
                ])

                precio = None
                if precio_entero:
                    if precio_decimal:
                        precio_texto = f"{precio_entero}.{precio_decimal}"
                    else:
                        precio_texto = precio_entero
                    precio = limpiar_precio(precio_texto)

                # -------- RATING --------
                rating = extraer_texto_seguro(producto, [
                    "span.a-icon-alt"
                ])

                # -------- NÚMERO DE REVIEWS --------
                numero_reviews = extraer_texto_seguro(producto, [
                    "span.a-size-base.s-underline-text",
                    "a[href*='customerReviews'] span.a-size-base",
                ])

                # Campos del detalle
                categoria = None
                marca = None
                disponibilidad = None
                descripcion = None

                # -------- ENTRAR AL DETALLE --------
                if url_producto:
                    abrir_producto_en_nueva_pestana(driver, url_producto)

                    detalle = capturar_detalle_producto(driver)
                    categoria = detalle["categoria"]
                    marca = detalle["marca"]
                    disponibilidad = detalle["disponibilidad"]
                    descripcion = detalle["descripcion"]

                    cerrar_pestana_actual(driver)

                # -------- REGISTRO FINAL --------
                registro = {
                    "nombre_producto": nombre_producto,
                    "precio": precio,
                    "rating": rating,
                    "numero_reviews": numero_reviews,
                    "categoria": categoria,
                    "marca": marca,
                    "disponibilidad": disponibilidad,
                    "descripcion": descripcion,
                    "grupo": NOMBRE_GRUPO,
                    "fuente": "Amazon",
                    "busqueda": BUSQUEDA,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                }

                if registro["nombre_producto"]:
                    datos_finales.append(registro)
                    print(f"Producto {i} agregado: {registro['nombre_producto'][:60]}")

            except Exception as e:
                print(f"Error en producto {i}: {e}")
                continue

        print(f"\nExtracción finalizada. Registros capturados: {len(datos_finales)}")
        return datos_finales

    except Exception as e:
        print(f"Error general en el scraper: {e}")
        return datos_finales

    finally:
        if driver:
            driver.quit()
            print("Navegador cerrado.")

In [12]:
# =========================
# 7: EJECUCIÓN DE PRUEBA
# =========================
# Aquí se ejecuta el scraper y se guarda el resultado en una variable.

datos = ejecutar_extraccion()

Abriendo Amazon...

ACCIÓN REQUERIDA:
1. Ve a http://localhost:6080/vnc.html
2. Revisa si Amazon muestra captcha o validación.
3. Si aparece, resuélvelo manualmente.
4. Cuando veas el listado de productos, vuelve aquí.



Presiona ENTER para continuar con la extracción... 


Sin bloqueo evidente. Iniciando scraping...
Bloques encontrados en el listado: 22
Producto 1 agregado: 15.6 Inch Laptop with Pentium Gold 6500Y(Up to 3.4GHz), 8GB 
Producto 2 agregado: Laptop Computer, Laptop with Gold 6500Y (Beat N5095, Up to 3
Producto 3 agregado: HP 14 Laptop, Intel Celeron N4020, 4 GB RAM, 64 GB Storage, 
Producto 4 agregado: KAIGERR Gaming Laptop, Laptops with AMD Ryzen 4300U(Up to 3.
Producto 5 agregado: Dell 15 Laptop DC15250-15.6-inch FHD 120Hz Display, Intel Co

Extracción finalizada. Registros capturados: 5
Navegador cerrado.


In [9]:
# =========================
# 8: REVISIÓN RÁPIDA
# =========================
# Esto permite ver cuántos registros se capturaron y cómo quedó el primero.

print(f"Total de registros capturados: {len(datos)}")

if datos:
    print("\nPrimer registro capturado:")
    print(datos[0])
else:
    print("No se capturaron datos.")

Total de registros capturados: 5

Primer registro capturado:
{'nombre_producto': '15.6 Inch Laptop Computer with Windows 11 Pro Laptops, 16GB RAM 1024GB SSD, Office 365, WiFi 6, Bluetooth 5.2, 180° Viewing, IPS FHD, 7000mAh, HDMI, Type-C, Cooling Fan(Gray)', 'precio': 329.99, 'rating': None, 'numero_reviews': None, 'categoria': None, 'marca': None, 'disponibilidad': None, 'descripcion': None, 'grupo': 'YHadfeg_Amazon', 'fuente': 'Amazon', 'busqueda': 'laptop', 'fecha_captura': '2026-04-23 19:12:46'}


In [10]:
# =========================
# 9: TABLA EN PANDAS
# =========================
# Convierte la lista de diccionarios en DataFrame para revisar mejor.

df = pd.DataFrame(datos)
df.head()

,nombre_producto,precio,rating,numero_reviews,categoria,marca,disponibilidad,descripcion,grupo,fuente,busqueda,fecha_captura
0,15.6 Inch Laptop Computer with Windows 11 Pro ...,329.99,None,None,None,None,None,None,YHadfeg_Amazon,Amazon,laptop,2026-04-23 19:12:46
1,"15.6 Inch Laptop Computer, 16GB RAM 512GB SSD,...",269.00,None,None,None,None,None,None,YHadfeg_Amazon,Amazon,laptop,2026-04-23 19:12:46
2,"KAIGERR Gaming Laptop, Laptops with AMD Ryzen ...",429.99,None,None,None,None,None,None,YHadfeg_Amazon,Amazon,laptop,2026-04-23 19:12:47
3,"HP 14 Laptop, Intel Celeron N4020, 4 GB RAM, 6...",194.95,None,None,None,None,None,None,YHadfeg_Amazon,Amazon,laptop,2026-04-23 19:12:47
4,Dell 15 Laptop DC15250-15.6-inch FHD 120Hz Dis...,NaN,None,None,None,None,None,None,YHadfeg_Amazon,Amazon,laptop,2026-04-23 19:12:47


In [11]:
# =========================
# 10: GUARDAR RESULTADO LOCAL
# =========================
# Guarda un archivo CSV para revisar rápidamente si la extracción quedó bien.

if not df.empty:
    df.to_csv("amazon_prueba_jupyter.csv", index=False, encoding="utf-8-sig")
    print("Archivo guardado: amazon_prueba_jupyter.csv")
else:
    print("No hay datos para guardar.")

Archivo guardado: amazon_prueba_jupyter.csv
